# Bakalarka aktual FINAL

Čistá finálna verzia notebooku pre bakalársku prácu.

**Odporúčaný postup spúšťania:**
1. Inštalácia a importy
2. Dataset a predspracovanie
3. Tréning a evaluácia
4. Experiment 3 – threshold tuning
5. Experiment 4 – Grad-CAM
6. Experiment 4b – audio saliency


In [ ]:

# 1. Inštalácia a importy
!pip install -q torch torchvision librosa scikit-learn matplotlib seaborn tqdm kagglehub

import os
import json
import random
import pickle
import numpy as np
import cv2
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
import pandas as pd

from tqdm import tqdm
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)


In [ ]:

# 2. Mount Google Drive a nastav cesty
from google.colab import drive
drive.mount('/content/drive')

EXP_DIR = "/content/drive/MyDrive/bakalarka_final"
os.makedirs(EXP_DIR, exist_ok=True)

SAMPLES_PATH = os.path.join(EXP_DIR, "samples.pkl")
MODEL_PATH = os.path.join(EXP_DIR, "best_model_v2.pth")
METRICS_PATH = os.path.join(EXP_DIR, "metrics_exp2.json")
RESULTS_DIR = os.path.join(EXP_DIR, "results")

os.makedirs(RESULTS_DIR, exist_ok=True)

print("SAMPLES_PATH =", SAMPLES_PATH)
print("MODEL_PATH   =", MODEL_PATH)
print("RESULTS_DIR  =", RESULTS_DIR)


In [ ]:

# 3. Dataset Celeb-DF v2
import kagglehub
path = kagglehub.dataset_download("reubensuju/celeb-df-v2")
print(f"Dataset: {path}")

def find_videos(base_path):
    videos = []
    for root, dirs, files in os.walk(base_path):
        for f in files:
            if f.endswith('.mp4'):
                videos.append(os.path.join(root, f))
    return videos

real_videos = find_videos(os.path.join(path, 'Celeb-real')) +               find_videos(os.path.join(path, 'YouTube-real'))
fake_videos = find_videos(os.path.join(path, 'Celeb-synthesis'))

random.seed(42)
real_subset = real_videos
fake_subset = random.sample(fake_videos, 890)

print(f"Reálnych: {len(real_subset)}, Fake: {len(fake_subset)}")


In [ ]:

# 4. Extrakcia tvárí a MFCC
!wget -q https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml -O /tmp/haarcascade.xml
face_cascade = cv2.CascadeClassifier('/tmp/haarcascade.xml')

def extract_faces(video_path, max_frames=10):
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total == 0:
        cap.release()
        return []

    indices = np.linspace(0, total - 1, max_frames, dtype=int)
    faces = []

    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret:
            continue

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        detections = face_cascade.detectMultiScale(
            gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30)
        )

        if len(detections) > 0:
            x, y, w, h = detections[0]
            m = 20
            x1 = max(0, x - m)
            y1 = max(0, y - m)
            x2 = min(frame.shape[1], x + w + m)
            y2 = min(frame.shape[0], y + h + m)

            face_crop = cv2.cvtColor(frame[y1:y2, x1:x2], cv2.COLOR_BGR2RGB)
            face_resized = cv2.resize(face_crop, (224, 224))
            face_tensor = torch.from_numpy(face_resized.copy()).permute(2, 0, 1).float() / 255.0
            faces.append(face_tensor)

    cap.release()
    return faces

def extract_mfcc(video_path, n_mfcc=40, max_len=128):
    audio_path = '/tmp/audio_temp.wav'
    os.system(f'ffmpeg -i "{video_path}" -q:a 0 -map a "{audio_path}" -y -loglevel quiet 2>/dev/null')
    if not os.path.exists(audio_path) or os.path.getsize(audio_path) == 0:
        return np.zeros((n_mfcc, max_len), dtype=np.float32)
    try:
        y, sr = librosa.load(audio_path, sr=16000, mono=True)
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
        if mfcc.shape[1] < max_len:
            mfcc = np.pad(mfcc, ((0, 0), (0, max_len - mfcc.shape[1])))
        else:
            mfcc = mfcc[:, :max_len]
    except Exception:
        return np.zeros((n_mfcc, max_len), dtype=np.float32)
    return mfcc.astype(np.float32)


In [ ]:

# 5. Predspracovanie datasetu a uloženie samples.pkl
all_samples = []

print("Spracovávam reálne videá...")
for vp in tqdm(real_subset):
    faces = extract_faces(vp, max_frames=10)
    if not faces:
        continue
    mfcc = extract_mfcc(vp)
    face = faces[len(faces) // 2]
    all_samples.append({'face': face, 'mfcc': mfcc, 'label': 0})

print("Spracovávam fake videá...")
for vp in tqdm(fake_subset):
    faces = extract_faces(vp, max_frames=10)
    if not faces:
        continue
    mfcc = extract_mfcc(vp)
    face = faces[len(faces) // 2]
    all_samples.append({'face': face, 'mfcc': mfcc, 'label': 1})

print(f"Celkovo vzoriek: {len(all_samples)}")

with open(SAMPLES_PATH, 'wb') as f:
    pickle.dump(all_samples, f)

print("Uložené:", SAMPLES_PATH)


In [ ]:

# 6. Dataset trieda, split a DataLoadery
with open(SAMPLES_PATH, 'rb') as f:
    all_samples = pickle.load(f)

print(f"Celkovo vzoriek: {len(all_samples)}")

train_s, temp_s = train_test_split(all_samples, test_size=0.3, random_state=42)
val_s,   test_s = train_test_split(temp_s,      test_size=0.5, random_state=42)

print(f"Train: {len(train_s)}, Val: {len(val_s)}, Test: {len(test_s)}")

augment = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
])

class DeepfakeDatasetAug(Dataset):
    def __init__(self, samples, augment_enabled=False):
        self.samples = samples
        self.augment_enabled = augment_enabled

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        face = s['face'].float()
        if self.augment_enabled:
            face = augment(face)
        mfcc = torch.tensor(s['mfcc']).float()
        label = torch.tensor(s['label']).float()
        return face, mfcc, label

train_loader = DataLoader(DeepfakeDatasetAug(train_s, augment_enabled=True), batch_size=16, shuffle=True)
val_loader   = DataLoader(DeepfakeDatasetAug(val_s, augment_enabled=False), batch_size=16)
test_loader  = DataLoader(DeepfakeDatasetAug(test_s, augment_enabled=False), batch_size=16)


In [ ]:

# 7. Definícia multimodálneho modelu
class VisualBranch(nn.Module):
    def __init__(self, output_dim=256):
        super().__init__()
        efficientnet = models.efficientnet_b0(weights='IMAGENET1K_V1')
        self.features = nn.Sequential(*list(efficientnet.children())[:-1])
        self.fc = nn.Linear(1280, output_dim)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.features(x)
        x = x.flatten(1)
        x = self.relu(self.fc(x))
        return x

class AudioBranch(nn.Module):
    def __init__(self, input_dim=40, hidden_dim=128, output_dim=256):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=2,
            batch_first=True,
            dropout=0.3
        )
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.permute(0, 2, 1)
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        out = self.relu(self.fc(out))
        return out

class DeepfakeDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.visual = VisualBranch(output_dim=256)
        self.audio  = AudioBranch(output_dim=256)
        self.fusion = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, face, mfcc):
        v = self.visual(face)
        a = self.audio(mfcc)
        combined = torch.cat([v, a], dim=1)
        return self.fusion(combined)

model = DeepfakeDetector().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Celkový počet parametrov: {total_params:,}")


In [ ]:

# 8. Tréning experimentu 2
model = DeepfakeDetector().to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=5e-5, weight_decay=1e-4)
criterion = nn.BCELoss()
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)

EPOCHS = 15
best_val_loss = float('inf')

train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    train_preds_epoch = []
    train_labels_epoch = []

    for face, mfcc, label in tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}'):
        face, mfcc, label = face.to(device), mfcc.to(device), label.to(device)

        optimizer.zero_grad()
        pred = model(face, mfcc).squeeze()
        loss = criterion(pred, label)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        pred_bin = (pred.detach() > 0.5).float()
        train_preds_epoch.extend(pred_bin.cpu().numpy().tolist())
        train_labels_epoch.extend(label.cpu().numpy().tolist())

    model.eval()
    val_loss = 0.0
    val_preds_epoch = []
    val_labels_epoch = []

    with torch.no_grad():
        for face, mfcc, label in val_loader:
            face, mfcc, label = face.to(device), mfcc.to(device), label.to(device)
            pred = model(face, mfcc).squeeze()
            loss = criterion(pred, label)
            val_loss += loss.item()

            pred_bin = (pred > 0.5).float()
            val_preds_epoch.extend(pred_bin.cpu().numpy().tolist())
            val_labels_epoch.extend(label.cpu().numpy().tolist())

    train_loss /= len(train_loader)
    val_loss /= len(val_loader)

    train_acc = accuracy_score(train_labels_epoch, train_preds_epoch)
    val_acc = accuracy_score(val_labels_epoch, val_preds_epoch)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    scheduler.step()

    print(
        f"Epoch {epoch+1}: "
        f"train_loss={train_loss:.4f}, val_loss={val_loss:.4f}, "
        f"train_acc={train_acc:.4f}, val_acc={val_acc:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), MODEL_PATH)
        print("  → Najlepší model uložený!")


In [ ]:

# 9. Evaluácia experimentu 2
model = DeepfakeDetector().to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()

all_preds, all_labels = [], []

with torch.no_grad():
    for face, mfcc, label in test_loader:
        face, mfcc = face.to(device), mfcc.to(device)
        pred = model(face, mfcc).squeeze().cpu().numpy()
        if np.isscalar(pred):
            pred = np.array([pred])
        all_preds.extend(pred.tolist())
        all_labels.extend(label.numpy().tolist())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels).astype(int)

threshold = 0.5
preds_binary = (all_preds > threshold).astype(int)

acc = accuracy_score(all_labels, preds_binary)
prec = precision_score(all_labels, preds_binary, zero_division=0)
rec = recall_score(all_labels, preds_binary, zero_division=0)
f1 = f1_score(all_labels, preds_binary, zero_division=0)
auc = roc_auc_score(all_labels, all_preds)

cm = confusion_matrix(all_labels, preds_binary)
tn, fp, fn, tp = cm.ravel()
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

metrics = {
    "threshold": threshold,
    "accuracy": round(acc, 4),
    "precision": round(prec, 4),
    "recall": round(rec, 4),
    "f1_score": round(f1, 4),
    "auc_roc": round(auc, 4),
    "specificity": round(specificity, 4),
    "true_negative": int(tn),
    "false_positive": int(fp),
    "false_negative": int(fn),
    "true_positive": int(tp),
    "n_total": int(len(all_labels)),
}

print("=== Výsledné metriky ===")
for k, v in metrics.items():
    print(f"{k}: {v}")

with open(METRICS_PATH, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)


In [ ]:

# 10. Grafy experimentu 2
# Loss curve
plt.figure(figsize=(7, 5))
plt.plot(train_losses, label='Train loss')
plt.plot(val_losses, label='Validation loss')
plt.xlabel('Epocha')
plt.ylabel('Loss')
plt.title('Priebeh učenia modelu')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'loss_curve_exp2.png'), dpi=200)
plt.show()

# Confusion matrix
plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    xticklabels=['Real', 'Fake'],
    yticklabels=['Real', 'Fake']
)
plt.xlabel('Predikovaná trieda')
plt.ylabel('Skutočná trieda')
plt.title('Matica zámen')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'confusion_matrix_exp2.png'), dpi=200)
plt.show()

# ROC curve
fpr, tpr, _ = roc_curve(all_labels, all_preds)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f'AUC = {auc:.4f}')
plt.plot([0, 1], [0, 1], linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC krivka')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'roc_curve_exp2.png'), dpi=200)
plt.show()

# Accuracy curve
plt.figure(figsize=(7, 5))
plt.plot(train_accs, label='Train accuracy')
plt.plot(val_accs, label='Validation accuracy')
plt.xlabel('Epocha')
plt.ylabel('Accuracy')
plt.title('Priebeh accuracy počas učenia')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'accuracy_curve_exp2.png'), dpi=200)
plt.show()

print("Grafy uložené v:", RESULTS_DIR)


In [ ]:

# 11. Experiment 3 – threshold tuning
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]
results = []

for threshold in thresholds:
    preds_binary = (all_preds > threshold).astype(int)

    acc_t = accuracy_score(all_labels, preds_binary)
    prec_t = precision_score(all_labels, preds_binary, zero_division=0)
    rec_t = recall_score(all_labels, preds_binary, zero_division=0)
    f1_t = f1_score(all_labels, preds_binary, zero_division=0)

    cm_t = confusion_matrix(all_labels, preds_binary)
    tn_t, fp_t, fn_t, tp_t = cm_t.ravel()
    specificity_t = tn_t / (tn_t + fp_t) if (tn_t + fp_t) > 0 else 0.0

    results.append({
        "threshold": threshold,
        "accuracy": round(acc_t, 4),
        "precision": round(prec_t, 4),
        "recall": round(rec_t, 4),
        "f1_score": round(f1_t, 4),
        "specificity": round(specificity_t, 4),
        "tn": int(tn_t),
        "fp": int(fp_t),
        "fn": int(fn_t),
        "tp": int(tp_t),
    })

results_df = pd.DataFrame(results)
print(results_df)

plt.figure(figsize=(7, 5))
plt.plot(results_df["threshold"], results_df["accuracy"], marker='o', label="Accuracy")
plt.plot(results_df["threshold"], results_df["precision"], marker='o', label="Precision")
plt.plot(results_df["threshold"], results_df["recall"], marker='o', label="Recall")
plt.plot(results_df["threshold"], results_df["f1_score"], marker='o', label="F1-score")
plt.plot(results_df["threshold"], results_df["specificity"], marker='o', label="Specificity")
plt.xlabel("Threshold")
plt.ylabel("Hodnota metriky")
plt.title("Vplyv rozhodovacieho prahu na výkon modelu")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'threshold_tuning.png'), dpi=200)
plt.show()

results_df.to_csv(os.path.join(RESULTS_DIR, 'threshold_tuning_results.csv'), index=False)


In [ ]:

# 12. Experiment 4 – Grad-CAM explainability
def tensor_to_numpy_image(face_tensor):
    img = face_tensor.detach().cpu().permute(1, 2, 0).numpy()
    img = np.clip(img, 0, 1)
    img = (img * 255).astype(np.uint8)
    return img

def overlay_heatmap_on_image(image_uint8, heatmap):
    heatmap_uint8 = np.uint8(255 * heatmap)
    heatmap_color = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)
    overlay = cv2.addWeighted(image_uint8, 0.6, heatmap_color, 0.4, 0)
    return heatmap_color, overlay

def generate_gradcam(model, face_tensor, mfcc_tensor, target_class=1):
    model.eval()
    activations = []
    gradients = []

    target_layer = model.visual.features[0][-1]

    def forward_hook(module, inp, out):
        activations.append(out)

    def backward_hook(module, grad_input, grad_output):
        gradients.append(grad_output[0])

    forward_handle = target_layer.register_forward_hook(forward_hook)
    backward_handle = target_layer.register_full_backward_hook(backward_hook)

    face_tensor = face_tensor.unsqueeze(0).to(device)
    mfcc_tensor = mfcc_tensor.unsqueeze(0).to(device)

    model.zero_grad()
    output = model(face_tensor, mfcc_tensor).squeeze()
    score = output if target_class == 1 else 1.0 - output
    score.backward()

    acts = activations[0]
    grads = gradients[0]
    weights = grads.mean(dim=(2, 3), keepdim=True)
    cam = (weights * acts).sum(dim=1, keepdim=True)
    cam = F.relu(cam)
    cam = cam.detach().cpu().numpy()

    if cam.ndim == 4:
        cam = cam[0, 0]
    elif cam.ndim == 3:
        cam = cam[0]

    cam = np.asarray(cam, dtype=np.float32)
    if cam.max() > 0:
        cam = cam / cam.max()

    cam = cv2.resize(cam, (224, 224), interpolation=cv2.INTER_LINEAR)

    forward_handle.remove()
    backward_handle.remove()

    return cam, float(output.detach().cpu().item())

def save_gradcam_for_sample(model, dataset_samples, sample_idx=0, target_class=None, save_dir=None):
    if save_dir is None:
        save_dir = os.path.join(RESULTS_DIR, "gradcam_results")
    os.makedirs(save_dir, exist_ok=True)

    sample = dataset_samples[sample_idx]
    face = sample['face'].float()
    mfcc = torch.tensor(sample['mfcc']).float()
    true_label = int(sample['label'])

    if target_class is None:
        target_class = true_label

    cam, pred_prob = generate_gradcam(model, face, mfcc, target_class=target_class)
    image_uint8 = tensor_to_numpy_image(face)
    heatmap_color, overlay = overlay_heatmap_on_image(image_uint8, cam)

    pred_label = 1 if pred_prob > 0.5 else 0
    label_map = {0: "Real", 1: "Fake"}

    fig = plt.figure(figsize=(12, 4))
    plt.subplot(1, 3, 1)
    plt.imshow(image_uint8)
    plt.title(f"Pôvodná tvár\nTrue: {label_map[true_label]}")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(heatmap_color)
    plt.title(f"Grad-CAM heatmap\nTarget: {label_map[target_class]}")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(overlay)
    plt.title(f"Overlay\nPred: {label_map[pred_label]} ({pred_prob:.3f})")
    plt.axis("off")

    plt.tight_layout()
    out_path = os.path.join(save_dir, f"gradcam_idx_{sample_idx}.png")
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"Uložené: {out_path}")


In [ ]:

# 13. Spustenie Grad-CAM na vybraných vzorkách
model = DeepfakeDetector().to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()

gradcam_dir = os.path.join(RESULTS_DIR, "gradcam_results")
os.makedirs(gradcam_dir, exist_ok=True)

# Vybrané vzorky podľa pôvodného experimentu
selected_gradcam_idxs = [0, 1, 3, 4, 6, 10]

for idx in selected_gradcam_idxs:
    save_gradcam_for_sample(model, test_s, sample_idx=idx, save_dir=gradcam_dir)


In [ ]:

# 14. Kompozitný obrázok pre Grad-CAM
img_paths = [
    os.path.join(RESULTS_DIR, "gradcam_results", "gradcam_idx_3.png"),
    os.path.join(RESULTS_DIR, "gradcam_results", "gradcam_idx_1.png"),
    os.path.join(RESULTS_DIR, "gradcam_results", "gradcam_idx_6.png"),
    os.path.join(RESULTS_DIR, "gradcam_results", "gradcam_idx_10.png"),
]

titles = [
    "a) Správne klasifikovaná deepfake vzorka",
    "b) Správne klasifikovaná reálna vzorka",
    "c) False positive",
    "d) False positive"
]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, img_path, title in zip(axes.ravel(), img_paths, titles):
    img = mpimg.imread(img_path)
    ax.imshow(img)
    ax.set_title(title, fontsize=12)
    ax.axis("off")

plt.tight_layout()
out_path = os.path.join(RESULTS_DIR, "gradcam_results", "gradcam_composite.png")
plt.savefig(out_path, dpi=250, bbox_inches="tight")
plt.show()
print(f"Uložené: {out_path}")


In [ ]:

# 15. Experiment 4b – audio saliency
def audio_saliency_for_sample(model, dataset, sample_idx=0, device='cuda'):
    model.eval()
    face, mfcc, label = dataset[sample_idx]

    face = face.unsqueeze(0).to(device)
    mfcc = mfcc.unsqueeze(0).to(device)
    mfcc.requires_grad_(True)

    output = model(face, mfcc).squeeze()

    model.zero_grad()
    output.backward()

    grads = mfcc.grad.detach().cpu().numpy()[0]
    mfcc_np = mfcc.detach().cpu().numpy()[0]

    saliency = np.abs(grads)
    saliency = saliency / (saliency.max() + 1e-8)

    pred_prob = output.detach().cpu().item()
    pred_label = 1 if pred_prob > 0.5 else 0

    true_name = "Fake" if int(label.item()) == 1 else "Real"
    pred_name = "Fake" if pred_label == 1 else "Real"

    print(f"Sample idx: {sample_idx}")
    print(f"True label: {true_name}")
    print(f"Pred label: {pred_name}")
    print(f"Prob fake:  {pred_prob:.4f}")

    return mfcc_np, saliency, true_name, pred_name, pred_prob

def plot_audio_saliency(mfcc_np, saliency, true_name, pred_name, pred_prob, save_path=None, sample_idx=None):
    plt.figure(figsize=(12, 8))
    plt.subplot(2, 1, 1)
    plt.imshow(mfcc_np, aspect='auto', origin='lower')
    plt.colorbar()
    plt.title(f"MFCC vstup | sample={sample_idx} | true={true_name} | pred={pred_name} | prob_fake={pred_prob:.4f}")
    plt.ylabel("MFCC koeficient")

    plt.subplot(2, 1, 2)
    plt.imshow(saliency, aspect='auto', origin='lower', cmap='hot')
    plt.colorbar()
    plt.title("Saliency mapa nad MFCC vstupom")
    plt.xlabel("Časový krok")
    plt.ylabel("MFCC koeficient")
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.show()

def plot_audio_saliency_time_importance(saliency, save_path=None, sample_idx=None, true_name=None, pred_name=None, pred_prob=None):
    time_importance = saliency.mean(axis=0)

    plt.figure(figsize=(10, 4))
    plt.plot(time_importance)
    plt.xlabel("Časový krok")
    plt.ylabel("Priemerná dôležitosť")
    plt.title(f"Časová dôležitosť | sample={sample_idx} | true={true_name} | pred={pred_name} | prob_fake={pred_prob:.4f}")
    plt.grid(True)
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.show()


In [ ]:

# 16. Výber reprezentatívnych vzoriek pre audio saliency
test_dataset_noaug = DeepfakeDatasetAug(test_s, augment_enabled=False)

correct_real = []
correct_fake = []
wrong_samples = []

model.eval()
with torch.no_grad():
    for idx in range(len(test_dataset_noaug)):
        face, mfcc, label = test_dataset_noaug[idx]
        face = face.unsqueeze(0).to(device)
        mfcc = mfcc.unsqueeze(0).to(device)

        prob_fake = model(face, mfcc).squeeze().item()
        pred = 1 if prob_fake > 0.5 else 0
        true = int(label.item())

        sample_info = {"idx": idx, "true": true, "pred": pred, "prob_fake": prob_fake}

        if pred == true:
            if true == 0 and len(correct_real) < 2:
                correct_real.append(sample_info)
            elif true == 1 and len(correct_fake) < 2:
                correct_fake.append(sample_info)
        else:
            if len(wrong_samples) < 2:
                wrong_samples.append(sample_info)

        if len(correct_real) >= 2 and len(correct_fake) >= 2 and len(wrong_samples) >= 2:
            break

print("=== Správne klasifikované REAL ===")
for s in correct_real:
    print(s)

print("\n=== Správne klasifikované FAKE ===")
for s in correct_fake:
    print(s)

print("\n=== Chybné vzorky ===")
for s in wrong_samples:
    print(s)


In [ ]:

# 17. Uloženie audio saliency máp a časových grafov
audio_saliency_dir = os.path.join(RESULTS_DIR, "audio_saliency_selected")
audio_time_dir = os.path.join(RESULTS_DIR, "audio_saliency_time_selected")
os.makedirs(audio_saliency_dir, exist_ok=True)
os.makedirs(audio_time_dir, exist_ok=True)

selected_samples = correct_real + correct_fake + wrong_samples

for s in selected_samples:
    idx = s["idx"]
    mfcc_np, saliency, true_name, pred_name, pred_prob = audio_saliency_for_sample(
        model=model,
        dataset=test_dataset_noaug,
        sample_idx=idx,
        device=device
    )

    save_path_1 = os.path.join(audio_saliency_dir, f"audio_saliency_idx_{idx}_true_{true_name}_pred_{pred_name}.png")
    save_path_2 = os.path.join(audio_time_dir, f"audio_time_idx_{idx}_true_{true_name}_pred_{pred_name}.png")

    plot_audio_saliency(
        mfcc_np, saliency, true_name, pred_name, pred_prob,
        save_path=save_path_1,
        sample_idx=idx
    )

    plot_audio_saliency_time_importance(
        saliency,
        save_path=save_path_2,
        sample_idx=idx,
        true_name=true_name,
        pred_name=pred_name,
        pred_prob=pred_prob
    )

print("Hotovo.")
print("Audio saliency:", audio_saliency_dir)
print("Audio time:", audio_time_dir)


In [ ]:

# 18. Kompozitné obrázky pre audio saliency
audio_saliency_paths = [
    os.path.join(audio_saliency_dir, "audio_saliency_idx_1_true_Real_pred_Real.png"),
    os.path.join(audio_saliency_dir, "audio_saliency_idx_3_true_Fake_pred_Fake.png"),
    os.path.join(audio_saliency_dir, "audio_saliency_idx_6_true_Real_pred_Fake.png"),
]

audio_saliency_titles = [
    "a) Správne klasifikovaná reálna vzorka",
    "b) Správne klasifikovaná deepfake vzorka",
    "c) Chybne klasifikovaná vzorka",
]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, img_path, title in zip(axes.ravel(), audio_saliency_paths, audio_saliency_titles):
    img = mpimg.imread(img_path)
    ax.imshow(img)
    ax.set_title(title, fontsize=11)
    ax.axis("off")

plt.tight_layout()
out_path = os.path.join(RESULTS_DIR, "audio_saliency_selected", "audio_saliency_composite.png")
plt.savefig(out_path, dpi=250, bbox_inches="tight")
plt.show()
print(f"Uložené: {out_path}")

audio_time_paths = [
    os.path.join(audio_time_dir, "audio_time_idx_1_true_Real_pred_Real.png"),
    os.path.join(audio_time_dir, "audio_time_idx_3_true_Fake_pred_Fake.png"),
    os.path.join(audio_time_dir, "audio_time_idx_6_true_Real_pred_Fake.png"),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, img_path, title in zip(axes.ravel(), audio_time_paths, audio_saliency_titles):
    img = mpimg.imread(img_path)
    ax.imshow(img)
    ax.set_title(title, fontsize=11)
    ax.axis("off")

plt.tight_layout()
out_path = os.path.join(RESULTS_DIR, "audio_saliency_time_selected", "audio_time_composite.png")
plt.savefig(out_path, dpi=250, bbox_inches="tight")
plt.show()
print(f"Uložené: {out_path}")
